# YOLOv11 Training on Google Colab

此 Notebook 用於在 Google Colab 上訓練 YOLOv11 模型。

## 前置作業
1.  將生成的資料集資料夾 `dataset_train_vaild_test` 壓縮成 `dataset_train_vaild_test.zip`。
2.  將 `dataset_train_vaild_test.zip` 上傳到您的 Google Drive (建議放在根目錄或指定資料夾)。
3.  確認 `data.yaml` 路徑設定正確。

## 1. 安裝 Ultralytics
安裝 YOLOv11 所需的套件。

In [1]:
%pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 21.2/107.7 GB disk)


## 2. 掛載 Google Drive
以便存取資料集並儲存訓練結果。

In [3]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: Interrupted by user

## 3. 解壓縮資料集
假設您的資料集位於 Drive 的 `MyDrive/dataset_train_vaild_test.zip`。
我們會將其解壓到 `/content/datasets` 以加快讀取速度。

In [ ]:
import os
import zipfile

# 定義路徑 (請依實際狀況修改)
zip_path = '/content/drive/MyDrive/Tibame_Project/data_picture/rename_dataset/dataset_train_vaild_test.zip' 
extract_path = '/content/drive/MyDrive/Tibame_Project/data_picture/rename_dataset'

if not os.path.exists(extract_path):
    os.makedirs(extract_path)

if os.path.exists(zip_path):
    print("正在解壓縮...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("解壓縮完成！")
else:
    print(f"錯誤：找不到檔案 {zip_path}，請檢查路徑。")

## 4. 開始訓練 (Training)
使用 YOLOv11m 進行微調。
請確保 `data` 參數指向解壓後正確的 `data.yaml` 位置。

In [ ]:
from ultralytics import YOLO

# 載入預訓練模型
model = YOLO('yolo11m.pt')

# 設定資料路徑 (注意：在 Colab 中通常是絕對路徑)
# 假設解壓後結構為 /content/datasets/dataset_train_vaild_test/data.yaml
# 請根據您的 zip 結構調整
yaml_path = '/content/datasets/dataset_train_vaild_test/data.yaml' 

# 開始訓練
results = model.train(
    data=yaml_path,
    epochs=50,      # 訓練輪數
    imgsz=640,      # 圖片大小
    batch=16,       # Batch size
    name='tibame_food_model', 
    device=0        # 使用 GPU
)

## 5. 備份模型權重
訓練完成後，將最佳權重複製回 Google Drive 以免 Colab 關閉後遺失。

In [ ]:
import shutil

source_path = '/content/runs/detect/tibame_food_model/weights/best.pt'
destination_path = '/content/drive/MyDrive/tibame_food_model_best.pt'

if os.path.exists(source_path):
    shutil.copy(source_path, destination_path)
    print(f"模型已備份至：{destination_path}")
else:
    print("找不到模型權重檔案。")